# 6.4 · Grid and implicit graphs

*Graphs* · 9 exercises

Graphs that are never written down: cells of a grid, moves of a piece, words one letter apart. Breadth-first search counts steps, depth-first search fills regions, and colours or memos keep each vertex to one visit.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-6.4) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-6.4) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-6.4)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 6.4.1 · Count the islands  `Core`

**Task.** `grid` is a list of strings, `#` land and `.` water. Write `count_islands(grid)` that returns how many islands there are, an island being land cells joined up, down, left or right. Scan the cells; at each unvisited land cell, count one island and visit all of it with a stack.

**Example.** `["##..#", "#...#", "..#..", "....#"]` -> `4`

**Big O.** O(R*C) time and space: each land cell is pushed once, when the island it belongs to is first found.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-1)

In [ ]:
# 6.4.1 Count the islands
def count_islands(grid):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_1():
    assert count_islands(['##..#', '#...#', '..#..', '....#']) == 4
    assert count_islands([]) == 0
    assert count_islands(['...']) == 0
    assert count_islands(['###', '#.#', '###']) == 1
    assert count_islands(['#.#', '.#.', '#.#']) == 5


test_ex_6_4_1()
print('6.4.1 passed')

### 6.4.2 · Can the graph be two-coloured?  `Core`

**Task.** Write `two_colourable(n, edges)` for vertices `0` to `n - 1` and undirected `[u, v]` edges: `True` when every vertex can be coloured one of two colours with no edge joining two of the same colour. Breadth-first search from each uncoloured vertex, giving neighbours the other colour.

**Example.** `4, [[0, 1], [1, 2], [2, 3], [3, 0]]` -> `True`; `3, [[0, 1], [1, 2], [2, 0]]` -> `False`

**Big O.** O(V + E): breadth-first search colours each vertex once, and an edge with both ends the same colour proves an odd cycle.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-2)

In [ ]:
# 6.4.2 Can the graph be two-coloured?
def two_colourable(n, edges):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_2():
    assert two_colourable(4, [[0, 1], [1, 2], [2, 3], [3, 0]])
    assert not two_colourable(3, [[0, 1], [1, 2], [2, 0]])
    assert two_colourable(5, [])
    assert not two_colourable(6, [[0, 1], [3, 4], [4, 5], [5, 3]])
    assert two_colourable(4, [[0, 1], [2, 3]])


test_ex_6_4_2()
print('6.4.2 passed')

### 6.4.3 · Copy a graph  `Core`

**Task.** Create `GraphNode(value)` with a `neighbours` list. Write `clone_graph(node)` that returns a deep copy of the connected graph reachable from `node` (`None` for `None`): new nodes with the same values and the same links, sharing nothing with the original. Keep a dictionary from original to copy and walk the graph.

**Example.** a ring `1 - 2 - 3 - 4 - 1` -> a new ring `1 - 2 - 3 - 4 - 1` that shares no node

**Big O.** O(V + E) time and space: a map from each original node to its copy makes sure every node is copied once, even around cycles.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-3)

In [ ]:
# 6.4.3 Copy a graph
class GraphNode:
    def __init__(self, value):
        raise NotImplementedError


def clone_graph(node):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_3():
    ring = [GraphNode(v) for v in (1, 2, 3, 4)]
    for i, node in enumerate(ring):
        node.neighbours = [ring[(i + 1) % 4], ring[(i - 1) % 4]]
    copy = clone_graph(ring[0])
    seen, stack = ({copy}, [copy])
    while stack:
        node = stack.pop()
        assert node not in ring
        for other in node.neighbours:
            if other not in seen:
                seen.add(other)
                stack.append(other)
    assert sorted((n.value for n in seen)) == [1, 2, 3, 4]
    assert [n.value for n in copy.neighbours] == [2, 4]
    assert copy.neighbours[0].neighbours[0].neighbours[0].value == 4
    assert copy.neighbours[0].neighbours[1] is copy
    assert clone_graph(None) is None
    alone = GraphNode(7)
    alone.neighbours = [alone]
    twin = clone_graph(alone)
    assert twin is not alone
    assert twin.neighbours == [twin]


test_ex_6_4_3()
print('6.4.3 passed')

### 6.4.4 · Largest island  `Practice`

**Task.** With the grid of 6.4.1, write `largest_island(grid)` that returns the number of cells in the biggest island (`0` when there is no land). Fill each island with a depth-first walk and count its cells.

**Example.** `["##..#", "#...#", "..#..", "....#"]` -> `3`

**Big O.** O(R*C) time and space: each cell is visited once, and each island's area is counted while it is filled.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-4)

In [ ]:
# 6.4.4 Largest island
def largest_island(grid):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_4():
    assert largest_island(['##..#', '#...#', '..#..', '....#']) == 3
    assert largest_island([]) == 0
    assert largest_island(['...']) == 0
    assert largest_island(['###', '#.#', '###']) == 8
    assert largest_island(['#.#', '.#.', '#.#']) == 1
    assert largest_island(['##.', '...', '###']) == 3


test_ex_6_4_4()
print('6.4.4 passed')

### 6.4.5 · Rounds until everything is reached  `Practice`

**Task.** `grid` holds `S` (a source), `.` (a cell to reach) and `#` (a wall). Each round, every reached cell reaches its 4 neighbours that are not walls. Write `rounds_to_spread(grid)` that returns how many rounds it takes to reach every `.` cell, `0` if there is none, or `-1` if some can never be reached. Start one breadth-first search from all sources together and go one level per round.

**Example.** `["S..#", ".#..", "...S"]` -> `2`

**Big O.** O(R*C) time and space: a breadth-first search started from every source at once reaches each cell in the fewest rounds.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-5)

In [ ]:
# 6.4.5 Rounds until everything is reached
def rounds_to_spread(grid):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_5():
    assert rounds_to_spread(['S..#', '.#..', '...S']) == 2
    assert rounds_to_spread(['S#.']) == -1
    assert rounds_to_spread(['S#']) == 0
    assert rounds_to_spread(['...']) == -1
    assert rounds_to_spread(['S....']) == 4
    assert rounds_to_spread(['S..', '#..', '..S']) == 2


test_ex_6_4_5()
print('6.4.5 passed')

### 6.4.6 · Cycle in a directed graph  `Practice`

**Task.** Write `has_cycle_directed(n, edges)` for vertices `0` to `n - 1` and directed `[u, v]` edges: `True` when the graph has a directed cycle. Depth-first search with three states per vertex; reaching a vertex that is still on the current path means a cycle (a finished one does not).

**Example.** `4, [[0, 1], [1, 2], [2, 0], [2, 3]]` -> `True`; `3, [[0, 1], [0, 2], [1, 2]]` -> `False`

**Big O.** O(V + E): with three colours (unvisited, on the current path, finished), an edge back to a vertex on the current path is exactly a cycle.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-6)

In [ ]:
# 6.4.6 Cycle in a directed graph
def has_cycle_directed(n, edges):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_6():
    assert has_cycle_directed(4, [[0, 1], [1, 2], [2, 0], [2, 3]])
    assert not has_cycle_directed(3, [[0, 1], [0, 2], [1, 2]])
    assert not has_cycle_directed(3, [])
    assert has_cycle_directed(2, [[1, 1]])
    assert has_cycle_directed(5, [[0, 1], [3, 4], [4, 2], [2, 3]])
    assert not has_cycle_directed(4, [[0, 1], [2, 1], [3, 2], [3, 0]])


test_ex_6_4_6()
print('6.4.6 passed')

### 6.4.7 · Word ladder  `Practice`

**Task.** Write `ladder_length(start, end, words)` that returns how many words are in the shortest chain from `start` to `end` (both counted) where each step changes exactly one letter and every word after `start` is in `words`; `0` when there is no chain. Breadth-first search, generating neighbours by trying each letter `a` to `z` at each position.

**Example.** `"cold", "warm", ["cord", "card", "ward", "warm", "word", "worm"]` -> `5` (cold, cord, card, ward, warm)

**Big O.** O(N * L * 26) for N words of length L: breadth-first search where each word's neighbours are found by trying every letter at every position against a set.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-7)

In [ ]:
# 6.4.7 Word ladder
def ladder_length(start, end, words):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_7():
    words = ['cord', 'card', 'ward', 'warm', 'word', 'worm']
    assert ladder_length('cold', 'warm', words) == 5
    assert ladder_length('cold', 'cord', words) == 2
    assert ladder_length('cold', 'warn', words) == 0
    assert ladder_length('ab', 'cd', ['ad', 'cd']) == 3
    assert ladder_length('ab', 'cd', ['xy', 'cd']) == 0


test_ex_6_4_7()
print('6.4.7 passed')

### 6.4.8 · Knight moves  `Practice`

**Task.** Write `knight_distance(n, start, goal)` that returns the fewest knight moves from `start` to `goal` on an `n` by `n` board (squares as `(row, col)`), or `-1` when the goal cannot be reached. Breadth-first search over squares.

**Example.** `8, (0, 0), (7, 7)` -> `6`

**Big O.** O(N^2) time and space on an N by N board: breadth-first search over squares, eight moves from each.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-8)

In [ ]:
# 6.4.8 Knight moves
def knight_distance(n, start, goal):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_8():
    assert knight_distance(8, (0, 0), (7, 7)) == 6
    assert knight_distance(8, (0, 0), (1, 2)) == 1
    assert knight_distance(8, (3, 3), (3, 3)) == 0
    assert knight_distance(3, (0, 0), (1, 1)) == -1
    assert knight_distance(8, (0, 0), (1, 1)) == 4
    assert knight_distance(1, (0, 0), (0, 0)) == 0


test_ex_6_4_8()
print('6.4.8 passed')

### 6.4.9 · Longest rising path in a grid  `Stretch`

**Task.** `grid` is a non-empty rectangle of integers. Write `longest_rising_path(grid)` that returns the number of cells on the longest path that moves up, down, left or right to a strictly larger value each step. A memoized depth-first search gives each cell `1 +` the best of its larger neighbours.

**Example.** `[[1, 2, 9], [5, 3, 8], [4, 6, 7]]` -> `7` (1, 2, 3, 6, 7, 8, 9)

**Big O.** O(R*C) time and space: rising steps can never loop, so the graph is acyclic, and memoizing each cell's best path means each is solved once.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-6-4-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-6-4-9)

In [ ]:
# 6.4.9 Longest rising path in a grid
def longest_rising_path(grid):
    raise NotImplementedError

In [ ]:
def test_ex_6_4_9():
    assert longest_rising_path([[1, 2, 9], [5, 3, 8], [4, 6, 7]]) == 7
    assert longest_rising_path([[7]]) == 1
    assert longest_rising_path([[1, 1], [1, 1]]) == 1
    assert longest_rising_path([[1, 2, 3, 4]]) == 4
    assert longest_rising_path([[5, 4], [2, 3]]) == 4


test_ex_6_4_9()
print('6.4.9 passed')